# Group Assignment 2: Building a CNN Model and Network Architecture

**Objective:**
Design and train an effective Convolutional Neural Network (CNN) for image classification on the CIFAR-10 dataset (10 object classes).

**Scope:**
This notebook goes beyond the minimum assignment requirements: in addition to building, summarizing, and compiling the architecture, we also train the model end-to-end on CIFAR-10, evaluate it on the held-out test set, and analyze per-class performance with a confusion matrix and misclassification examples.

**Design Philosophy:**
Rather than copy the reference architecture verbatim (single Conv2D per block, no normalization, no regularization), we design a **VGG-inspired architecture** with three principled improvements:
1. **Double Conv2D blocks** to increase representational depth without aggressive spatial downsampling.
2. **Batch Normalization** after every Conv2D to stabilize training.
3. **Progressive Dropout + L2 weight decay** as a layered regularization strategy against overfitting on the small (32x32) input.


### Section 0: Setup and Reproducibility

We define a single `SEED` constant and pass it to `train_test_split` to guarantee that the train/validation split is identical across runs.

**Scope of reproducibility (read this carefully):**
- Reproducible: which samples land in `X_train` vs `X_val`.
- *Not* reproducible: weight initialization, dropout masks, augmentation randomness. Therefore the final test accuracy will fluctuate by roughly +/-1% across runs, but the data partitioning is fixed.

We deliberately do not enable `tf.config.experimental.enable_op_determinism()` because the 1.5x-2x training slowdown is not justified for an academic assignment.


In [ ]:
SEED = 42

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available:      {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Random seed:        {SEED}")


### Section 1: Load CIFAR-10 Dataset

CIFAR-10 contains 60,000 32x32 RGB images across 10 classes, with 50,000 training and 10,000 test samples (the test split is fixed by the dataset authors and we keep it untouched throughout this notebook).


In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Data loaded successfully!")
print(f"Number of training samples: {len(train_images)}")
print(f"Number of test samples:     {len(test_images)}")
print(f"Image shape:                {train_images.shape[1:]}")
print(f"Number of classes:          {len(class_names)}")


### Section 2: Preprocessing and Stratified Train/Validation Split

Three preprocessing steps:

1. **Pixel normalization (0-1).** Scale 8-bit pixel values from [0, 255] to [0, 1] for numerical stability and to align with the implicit assumptions of the He/Glorot weight initializers.
2. **Stratified train/validation split.** Carve a validation set out of the 50,000 training samples using `sklearn.model_selection.train_test_split` with `stratify=y_train` to ensure all 10 classes are equally represented in both splits. We do *not* touch the original 10,000 test samples.
3. **One-hot label encoding.** Convert integer labels (0-9) to 10-dimensional one-hot vectors, as required by `categorical_crossentropy`.

**Important sequencing note:** stratification must use *integer* labels, not one-hot. Therefore we split first, then one-hot encode.


In [ ]:
# 1. Normalize pixel values to [0, 1]
X_train_full = train_images.astype('float32') / 255.0
X_test       = test_images.astype('float32') / 255.0
y_train_full = train_labels.flatten()   # Keep as integers for stratification
y_test_int   = test_labels.flatten()

# 2. Stratified train/validation split (10% held out for validation)
X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_full
)

# 3. One-hot encode labels for categorical_crossentropy
y_train = to_categorical(y_train_int, num_classes=10)
y_val   = to_categorical(y_val_int,   num_classes=10)
y_test  = to_categorical(y_test_int,  num_classes=10)

print("Split and preprocessing complete.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")
print(f"\nClass distribution in validation set (should be ~500 per class):")
unique, counts = np.unique(y_val_int, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  {class_names[cls]:12s}: {count}")


### Section 3: Data Augmentation Pipeline

Augmentation is applied via a `tf.data` pipeline rather than as layers inside the model. Two reasons:

1. **Keeps `model.summary()` clean.** The assignment focuses on architecture, so augmentation should not appear as architecture layers.
2. **Augmentation is applied only to the training set.** Validation and test data must be evaluated on un-augmented images, otherwise the metrics become a noisy moving target rather than a true measure of generalization.

We use mild augmentation (small rotation and zoom, horizontal flip) because aggressive augmentation on 32x32 images destroys discriminative features. This is a lesson directly carried over from the previous (cat-vs-dog) assignment, where heavy augmentation caused training accuracy to stall around 50%.


In [ ]:
# Augmentation pipeline (only applied to training data)
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name='augmenter')

batch_size = 64

# Build tf.data datasets
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10000, seed=SEED)
            .batch(batch_size)
            .map(lambda x, y: (augmenter(x, training=True), y),
                 num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(batch_size)
          .prefetch(tf.data.AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(batch_size)
           .prefetch(tf.data.AUTOTUNE))

print("Datasets ready:")
print(f"  train_ds: {len(X_train)} samples, augmented")
print(f"  val_ds:   {len(X_val)} samples, no augmentation")
print(f"  test_ds:  {len(X_test)} samples, no augmentation")


### Section 4: Building the CNN Architecture

Sequential CNN with three convolutional blocks followed by a fully-connected classifier. Each block doubles filter count (32 -> 64 -> 128) while halving spatial dimensions via MaxPooling, following the standard VGG design pattern.

**Block structure:**
`Conv2D -> BatchNorm -> Conv2D -> BatchNorm -> MaxPooling2D -> Dropout`

**Regularization stack:**
- L2 weight decay (`1e-4`) on every Conv2D and Dense layer
- Batch Normalization after every Conv2D and after the dense classifier layer
- Progressive Dropout: 0.2 -> 0.3 -> 0.4 -> 0.5 (deeper layers receive stronger regularization)


In [ ]:
weight_decay = 1e-4

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # Block 1: 32 filters, 32x32 -> 16x16
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Block 2: 64 filters, 16x16 -> 8x8
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 3: 128 filters, 8x8 -> 4x4
    layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')   # 10 outputs for object classification
])


### Section 5: Display the Model Architecture

In [ ]:
model.summary()

### Section 6: Compile the Model

Compile with the **Adam** optimizer (adaptive learning rate, robust default for vision) and **categorical_crossentropy** loss (the natural choice for multi-class classification with one-hot labels and a softmax output).


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")
print(f"Optimizer: {model.optimizer.__class__.__name__}")
print(f"Loss:      {model.loss}")
print(f"Metrics:   {[m.name for m in model.metrics]}")


### Section 7: Training the Model

Two callbacks govern the training loop:

- **EarlyStopping** (`patience=10`, `restore_best_weights=True`): if validation loss does not improve for 10 consecutive epochs, training stops and the model weights are rolled back to the best validation epoch. The patience is set generously to give `ReduceLROnPlateau` time to act first.
- **ReduceLROnPlateau** (`patience=3`, `factor=0.5`): if validation loss plateaus for 3 epochs, the learning rate is halved (down to a floor of `1e-6`). This lets the model fine-tune in smaller steps once the loss landscape flattens.

We set `epochs=50` as an upper bound; EarlyStopping will typically halt training between epochs 25-35.


In [ ]:
callback_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        patience=3,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callback_list,
    verbose=1
)

print(f"\nTraining finished after {len(history.history['loss'])} epochs.")
print(f"Best validation loss:     {min(history.history['val_loss']):.4f}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")


### Section 8: Training History

Plot accuracy and loss for both training and validation sets across epochs. The shape of these curves is the primary diagnostic for overfitting (val curve diverging upward from train curve in loss, downward in accuracy) or underfitting (both curves flat at low accuracy).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Loss curve
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Section 9: Test Set Evaluation

The validation set guided training (early stopping, learning rate decay) and is therefore *not* an unbiased estimate of generalization. The 10,000-sample test set, untouched until now, is.

We report two layers of metrics:
1. **Aggregate**: test loss and overall accuracy.
2. **Per-class**: precision, recall, and F1-score for each of the 10 classes via `classification_report`. This reveals which classes the model handles well and which it struggles with.


In [ ]:
# Aggregate metrics
print("Evaluating on the test set...")
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

# Per-class metrics
y_pred_proba = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test_int

print("Per-class classification report:")
print("-" * 65)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


### Section 10: Confusion Matrix

The 10x10 confusion matrix exposes which class pairs the model confuses. CIFAR-10 has well-known difficult pairs:

- **cat vs. dog**: similar fur textures and poses
- **automobile vs. truck**: both are land vehicles with similar shapes
- **deer vs. horse**: both are four-legged animals in outdoor settings
- **bird vs. airplane**: both appear against sky backgrounds

The diagonal shows correct predictions; off-diagonal cells reveal systematic confusions.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - CIFAR-10 Test Set')
plt.tight_layout()
plt.show()


### Section 11: High-Confidence Misclassifications

Rather than visualize random errors, we surface the **9 test images where the model was most confident in a wrong prediction** (highest predicted probability among misclassified samples).

These cases are diagnostically more interesting than random errors: they reveal *systematic* failures of the model's learned representations rather than borderline noise. Each title shows `True -> Predicted (confidence)`.


In [ ]:
# Identify all misclassified samples
misclassified_mask = (y_pred != y_true)
misclassified_indices = np.where(misclassified_mask)[0]

# For each misclassified sample, get the confidence in the (wrong) predicted class
predicted_confidence = y_pred_proba[misclassified_indices, y_pred[misclassified_indices]]

# Sort by confidence descending and take top 9
top9_local_idx = np.argsort(predicted_confidence)[-9:][::-1]
top9_global_idx = misclassified_indices[top9_local_idx]

# Visualize
plt.figure(figsize=(12, 12))
for i, idx in enumerate(top9_global_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx])
    true_class = class_names[y_true[idx]]
    pred_class = class_names[y_pred[idx]]
    confidence = y_pred_proba[idx, y_pred[idx]]
    plt.title(f"{true_class} -> {pred_class}\n(confidence: {confidence:.1%})",
              fontsize=11)
    plt.axis('off')

plt.suptitle('Top 9 Most Confident Misclassifications', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


### Section 12: Architecture Design Report

This section explains the three core design decisions required by the assignment: **number of layers**, **kernel size**, and **use of dropout**.

---

#### 12.1 Choice of Number of Layers

The architecture uses **three convolutional blocks** with **two Conv2D layers per block** (six convolutional layers in total), plus a fully-connected classifier head. The reasoning:

- **Spatial constraint from input size.** CIFAR-10 images are only 32x32. Each MaxPooling2D layer halves the spatial dimensions: 32 -> 16 -> 8 -> 4. Adding a fourth block would reduce the feature map to 2x2, leaving too little spatial information for meaningful convolution. Three blocks is therefore the principled upper bound for this input size.
- **Double Conv2D per block (VGG pattern).** Stacking two 3x3 convolutions before pooling gives an effective receptive field of 5x5 with fewer parameters and one extra non-linearity compared to a single 5x5 convolution. This is the same reasoning that motivated VGGNet's design and is well-suited to small images where features are local.
- **Progressive filter growth (32 -> 64 -> 128).** Doubling filter counts as spatial dimensions shrink keeps the per-layer computational cost roughly balanced while allowing deeper layers to capture more abstract feature combinations.
- **Lightweight classifier head.** A single Dense(128) layer is sufficient because the 4x4x128 = 2048 flattened feature vector is already a compact, high-level representation. Stacking multiple large Dense layers tends to overfit on CIFAR-10 without proportional accuracy gains.

#### 12.2 Kernel Size

All convolutional layers use a **3x3 kernel**. This is a deliberate choice over larger kernels (5x5, 7x7) for three reasons:

- **Parameter efficiency.** A 3x3 kernel with C input/output channels has 9C^2 parameters, versus 25C^2 for a 5x5 kernel. Two stacked 3x3 layers (18C^2 parameters) achieve the same 5x5 receptive field with fewer parameters and one additional ReLU non-linearity.
- **Fine-grained features.** On 32x32 inputs, the discriminative features (edges of an animal's ear, the silhouette of a wing) are inherently small. A 3x3 kernel matches this scale without averaging over too much surrounding context.
- **Empirical convention.** The 3x3 kernel has become the de facto standard since VGGNet (2014) and remains dominant in modern architectures (ResNet, EfficientNet). It is the safe and well-validated default.

#### 12.3 Use of Dropout

Dropout is applied at four points with **progressively increasing rates** (0.2, 0.3, 0.4, 0.5):

- **Why progressive rates?** Earlier convolutional layers learn generic low-level features (edges, colors, textures) that are useful across all classes; aggressive dropout here would discard signal that the rest of the network depends on. Deeper layers learn more specialized, class-specific patterns that are also more prone to memorization, so they tolerate (and benefit from) heavier dropout.
- **Why 0.5 in the Dense layer?** The classifier head has the highest parameter density (the Flatten -> Dense(128) connection alone has ~262k parameters, almost half the model's total). This is where overfitting risk is highest, so the original 0.5 dropout rate proposed by Srivastava et al. (2014) is appropriate.
- **Dropout placement: after pooling, before the next block.** Placing dropout after MaxPooling2D (rather than between the two Conv2D layers) avoids disrupting the receptive-field accumulation within a block. The dropout acts as a regularization gate between blocks, not within them.
- **Complementary to other regularizers.** Dropout is paired with L2 weight decay (`1e-4`) and BatchNormalization. The three serve different purposes: L2 penalizes large weights, BatchNorm stabilizes activations, Dropout forces redundancy in feature representations. Together they form a layered defense against overfitting that no single technique provides alone.


### Section 13: Training Results Discussion

> **Note:** This section is a discussion framework. After training completes, fill in the bracketed `[...]` placeholders with your actual numbers, and select the bullets in each subsection that match your observations.

#### 13.1 Convergence Behavior

- **Final test accuracy:** `[XX.X%]` (filled in from Section 9).
- **Number of epochs trained:** `[NN]` (training stopped via EarlyStopping at epoch `[NN]`).
- **Train vs. validation gap:** `[gap%]` at the final epoch. A small gap (<5%) indicates the regularization stack (Dropout + L2 + BatchNorm + augmentation) is working as designed; a large gap (>10%) would indicate the model is still overfitting despite regularization.
- **Effect of `ReduceLROnPlateau`:** the learning rate dropped at epoch `[NN]`, after which validation loss [continued to improve / plateaued]. This confirms that smaller steps were [valuable / not the bottleneck] in the late training phase.

#### 13.2 Per-Class Performance

From the classification report and confusion matrix:

- **Best-performing classes:** typically `automobile`, `ship`, `truck` (vehicle classes have rigid geometry and distinctive backgrounds). Confirm whether your run matches.
- **Worst-performing classes:** typically `cat`, `dog`, `bird`. Animal classes share fur/feather textures, similar poses, and natural backgrounds.
- **Strongest confusion pairs in the matrix:** `[e.g., cat -> dog with NN errors, deer -> horse with NN errors]`. These align with known CIFAR-10 difficulty patterns.

#### 13.3 Insights from High-Confidence Misclassifications

The Section 11 visualization typically shows:

- **Pose ambiguity:** an animal photographed at an unusual angle resembles another species in silhouette.
- **Background dominance:** some samples are misclassified primarily on background cues (sky -> bird/airplane confusion, water -> ship confusion).
- **Color cues misleading the model:** a black cat may be classified as dog if the training set's cats were lighter-colored on average.

#### 13.4 Improvements for Future Iterations

Concrete next steps that would likely lift accuracy beyond the current `[XX.X%]`:

- **Transfer learning** with a pretrained backbone (MobileNetV2, ResNet50). Requires upscaling 32x32 inputs to 96x96 or larger to meet pretrained model input requirements.
- **Stronger augmentation** (Mixup, CutMix, RandAugment). The current mild augmentation is conservative; once the model is regularized enough, these techniques can extract additional generalization.
- **Architecture upgrades**: replace `Flatten + Dense` with `GlobalAveragePooling2D`, which dramatically reduces parameters and tends to improve generalization on small datasets.
- **Ensemble** of 3-5 independently trained models (different seeds), averaging predictions. Typically adds 1-2% test accuracy at the cost of training time.
